In [ ]:
import numpy as np

import matplotlib.pyplot as plt
import astropy.units as u
from astropy.constants import G, c, M_sun, sigma_sb, h, k_B

from scipy.integrate import quad
from scipy.interpolate import interp1d

# G = 6.67430e-11  # m^3 kg^-1 s^-2
# c = 2.99792458e10  # cm/s
# h = 6.62607015e-27  # erg s
# k = 1.380649e-16  # erg/K
# G = gravitational constant, c = speed of light, h = Plank constant, k = Boltzmann constant, sigma = Stefan-Boltzmann constant

# sigma = 5.670374419e-5  # erg cm^-2 s^-1 K^-4 [stefan-boltzmann constant]
# M_sun = 1.9847e33   # g



import matplotlib.ticker as ticker
from scipy.interpolate import interp1d
plt.rcParams["font.family"] = "Times New Roman" #font

In [ ]:
#defining some physical parameters to debug 

pc = 3.085677581e18  # cm
kpc = 3.085677581e21  # cm


M = 1.4 * M_sun
Mdot = 1e17 * u.g / u.s

R_in = 12e5 * u.cm
R_out = 1000e5 * u.cm

inclination = np.radians(45)

D = 5 * u.kpc

In [ ]:
def disk_temp(R, M, Mdot, R_in):
    term = (3 * G * M * Mdot) / (8 * np.pi * R**3 * sigma_sb)
    # R = radius of temp, M = NS mass, Mdot = mass accretion rate, R_in = inner edge
    boundary = 1 - np.sqrt(R_in / R)

    return (term * boundary)**0.25

In [ ]:
# plotting the temp based on R

#random radii
R_values = np.logspace(
    np.log10((R_in * 1.001).to(u.cm).value),
    np.log10((R_out).to(u.cm).value),
    500
) * u.cm

T_values = disk_temp(R_values, M, Mdot, R_in)

#plotting time

plt.figure(figsize=(7, 5))
plt.loglog(
    R_values, 
    T_values, 
    label = "Temperature vs. Radius", 
    linewidth = 2, 
    color = "pink"
    ) #legend


plt.xlabel("Radius [cm]", fontsize = 15)
plt.ylabel("Temperature [K]", fontsize = 15)
plt.title("Accretion Disk Temp", fontsize = 20)
plt.legend()
plt.minorticks_on()

#tick stuff
plt.grid(True, which = "both", ls = "--", lw = 0.5, alpha = 0.4)
plt.tick_params(axis = "both", which = "both", top = True, right = True, direction = "inout" , length = 3, width = 1, size = 4)

ax = plt.gca()
ax.xaxis.set_major_locator(ticker.LogLocator(base = 10, subs = np.arange(1.0, 10.0) * 0.1, numticks = 3))
ax.yaxis.set_major_locator(ticker.LogLocator(base = 10, subs = np.arange(1.0, 10.0) * 0.1, numticks = 3))
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())

plt.ticklabel_format(axis = "both", style = "sci")
#end of tick stuff

plt.gcf().canvas.draw()
for i, j in enumerate(ax.xaxis.get_ticklabels()):
    if i % 2 != 0:
        j.set_y(j.get_position()[1] - 0.03)

# makes the ticks alternate

plt.show()

# for future reference: this was kinda giving me a hard time... it kept having unit errors due tot he astropy and would work once when I ran and literally not the next time after not chnaging anything? 


In [ ]:
# adding in frequency

def plank_nu(nu, T):
    exponent = (h * nu / (k_B * T)).decompose().value #makes exponent dimensionless

    return (2 * h * nu**3 / c**2) / np.expm1(exponent)

# nu = frequency in Hertz, T = temp in K

In [ ]:
def disk_integrand(R, nu, M, Mdot, R_in):
    T = disk_temp(R, M, Mdot, R_in)

    return 2 * np.pi * R * plank_nu(nu, T)

# ok so that's the stuff going into the integral done

# actual flux fn:

def disk_flux_nu(nu, M, Mdot, R_in, R_out, inclination, D):

    def integrand_for_quad(R_value):
        value = R_value * u.cm
        value = disk_integrand(value, nu, M, Mdot, R_in)

        return value.value # Return the value without units for integration
    
    integral, error = quad(
        integrand_for_quad,
        R_in.to(u.cm).value,
        R_out.to(u.cm).value,
    )

    sample_unit = disk_integrand(
    R_in * 1.001,
    nu,
    M,
    Mdot,
    R_in    
).unit

    integral = integral * sample_unit * u.cm # Reapply the unit after integration

    F_nu = (np.cos(inclination) / D**2) * integral
    return F_nu

# make sure inclination is in radians when running
#use:
    # inclination = np.radians(angle)
    # np.cos(inclination)


In [ ]:
from scipy.integrate import dblquad

def disk_bol_L(M, Mdot, R_in, R_out):

    def integrand_for_dblquad(R_value, nu_value):
        #reattatching the physical units
        R = R_value * u.cm
        nu = nu_value * u.Hz

        value = disk_integrand(R, nu, M, Mdot, R_in)

        return value.to_value(
            u.erg / (u.s * u.Hz * u.cm)
        )

    integral, error = dblquad(
        integrand_for_dblquad, #tbh I am using chat to debug here bc I was having issues and I'm not totally sure what it's telling me to do
        1e10, 1e20, # frequency range in Hz
        lambda nu: R_in.to_value(u.cm), # lower limit of R
        lambda nu: R_out.to_value(u.cm)  # upper limit of R
    )

    #unit of og integrand
    sample_unit = disk_integrand(
        R_in * 1.001,
        1e17 * u.Hz,
        M,
        Mdot,
        R_in    
    ).unit

    integral = integral * sample_unit * u.cm * u.Hz # Reapply the unit after integration
    L_bol = integral * 4 * np.pi**2  # multiply by 4pi^2 for total luminosity
    return L_bol  # returning the total bolometric luminosity


In [ ]:
#testing the flux fn
nu = 1e17 * u.Hz        

inclination = np.radians(45)

F_nu = disk_flux_nu( 
    nu,
    M,
    Mdot,
    R_in,
    R_out,
    inclination,
    D
    )



F_bol = disk_bol_L( 
    M,
    Mdot,
    R_in,
    R_out,
    )

L_acc = (0.1 * Mdot * c**2).to(u.erg / u.s) # converting to luminosity units

print("Accretion luminosity:", L_acc) # finally working !!!!!!!!
print("Flux at nu:", F_nu)
print("Bolometric flux:", F_bol)

# seems like the order of mag is more reasonable? I had 10^38 in my notes but this is givign 10^36. is it still ballpark?

In [ ]:
# time to try a spectrum!

#generating frequencies
frequencies = np.logspace(14, 19, 200)  # in Hz obvi

fluxes = []

for nu in frequencies:
    F = disk_flux_nu(
        nu * u.Hz,
        M,
        Mdot,
        R_in,
        R_out,
        inclination,
        D
    )
    fluxes.append(F.to_value(u.erg / (u.cm**2 * u.s * u.Hz)))  # Convert to desired units

fluxes = np.array(fluxes)

#plotting time

plt.figure(figsize=(7, 5))
plt.loglog(
    frequencies, 
    fluxes, 
    label = "Flux vs Frequency", 
    linewidth = 2, 
    color = "pink"
    ) #legend(ary)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Flux (ergs/cm²/s/Hz)")
plt.title("Disk Spectrum")
plt.legend()

plt.minorticks_on()


# grid
plt.grid(
    True,
    which="both",
    ls="--",
    lw=0.5,
    alpha=0.4
)

# tick styling
plt.tick_params(
    axis="both",
    which="both",
    top=True,
    right=True,
    direction="inout",
    length=3,
    width=1
)

ax = plt.gca()


# log tick locations
ax.xaxis.set_major_locator(
    ticker.LogLocator(
        base=10,
        numticks=6
    )
)

ax.yaxis.set_major_locator(
    ticker.LogLocator(
        base=10,
        numticks=6
    )
)

# scientific notation labels
ax.xaxis.set_major_formatter(
    ticker.LogFormatterSciNotation(base=10)
)

ax.yaxis.set_major_formatter(
    ticker.LogFormatterSciNotation(base=10)
)

# makes graph fit nicely
plt.tight_layout()

plt.show()

In [ ]:
#varying NS mass

#ex
masses = [1.2, 1.4, 1.8, 2.0]

for mass in masses:
    M = mass * M_sun
    fluxes = []
    for nu in frequencies:
        F = disk_flux_nu(
            nu * u.Hz,
            M,
            Mdot,
            R_in,
            R_out,
            inclination,
            D
        )
        fluxes.append(F.to_value(u.erg / (u.cm**2 * u.s * u.Hz)))  # Convert to desired units

    fluxes = np.array(fluxes)

    plt.loglog(
        frequencies, 
        fluxes, 
        label = f"Mass = {mass} M_sun", 
        linewidth = 2
        )

# labels
plt.xlabel(r"Frequency $\nu$ [Hz]", fontsize=15)

plt.ylabel(
    r"$F_\nu$ [erg cm$^{-2}$ s$^{-1}$ Hz$^{-1}$]",
    fontsize=15
)

plt.title(
    "Disk Spectrum for Different NS Masses",
    fontsize=18
)


# legend
plt.legend(
    fontsize=11,
    frameon=True
)


# minor ticks
plt.minorticks_on()


# grid stuff
plt.grid(
    True,
    which="both",
    ls="--",
    lw=0.5,
    alpha=0.4
)

# tick styling
plt.tick_params(
    axis="both",
    which="both",
    top=True,
    right=True,
    direction="in",
    length=5,
    width=1
)


# get current axes
ax = plt.gca()


# x-axis log ticks
ax.xaxis.set_major_locator(
    ticker.LogLocator(
        base=10,
        numticks=6
    )
)

# y-axis log ticks
ax.yaxis.set_major_locator(
    ticker.LogLocator(
        base=10,
        numticks=6
    )
)

# formatting as 10^x
ax.xaxis.set_major_formatter(
    ticker.LogFormatterSciNotation(base=10)
)

ax.yaxis.set_major_formatter(
    ticker.LogFormatterSciNotation(base=10)
)


# increasing tick label size
ax.tick_params(
    axis="both",
    labelsize=11
)


# making sure labels don't get cut off
plt.tight_layout()

plt.show()

In [ ]:
# this is the same as above but I'm attempting to make it more readable? idk it looks supa tight right 
# now so I am consulting the internet for help but not sure if this is a good official practice

# varying NS mass (pt. 2)

masses = [1.2, 1.4, 1.8, 2.0]

# dictionary to store each spectrum
spectra = {}

plt.figure(figsize=(8, 6))

for mass in masses:

    M = mass * M_sun

    fluxes = []

    for nu in frequencies:

        F = disk_flux_nu(
            nu * u.Hz,
            M,
            Mdot,
            R_in,
            R_out,
            inclination,
            D
        )

        #convert units
        F_cgs = F.to_value(u.erg / (u.cm**2 * u.s * u.Hz))  

        fluxes.append(F_cgs)

    fluxes_cgs = np.array(fluxes)

   
    # save spectrum so we can use it later
    spectra[mass] = fluxes_cgs

    # plot spectrum
    plt.loglog(
        frequencies,
        fluxes_cgs,
        label=fr"$M = {mass}\,M_\odot$",
        linewidth=2
    )


# labels
plt.xlabel(r"Frequency $\nu$ [Hz]", fontsize=15)

plt.ylabel(
    r"$F_\nu$ [erg cm$^{-2}$ s$^{-1}$ Hz$^{-1}$]",
    fontsize=15
)

plt.title(
    "Disk Spectrum for Different NS Masses",
    fontsize=18
)

plt.legend(fontsize=11)

plt.minorticks_on()

plt.grid(
    True,
    which="both",
    ls="--",
    lw=0.5,
    alpha=0.4
)

plt.tick_params(
    axis="both",
    which="both",
    top=True,
    right=True,
    direction="in",
    length=5,
    width=1
)

ax = plt.gca()

ax.xaxis.set_major_locator(
    ticker.LogLocator(base=10, numticks=6)
)

ax.yaxis.set_major_locator(
    ticker.LogLocator(base=10, numticks=6)
)

ax.xaxis.set_major_formatter(
    ticker.LogFormatterSciNotation(base=10)
)

ax.yaxis.set_major_formatter(
    ticker.LogFormatterSciNotation(base=10)
)

ax.tick_params(
    axis="both",
    labelsize=11
)

plt.tight_layout()

plt.show()


# flux ratio relative to 1.4 solar masses bc it is allegedly a good reference pt...

reference_mass = 1.4

reference_flux = spectra[reference_mass]

plt.figure(figsize=(8, 6))

for mass in masses:

    flux_ratio = spectra[mass] / reference_flux

    plt.semilogx(
        frequencies,
        flux_ratio,
        label=fr"$M = {mass}\,M_\odot$",
        linewidth=2
    )


# reference line
plt.axhline(
    y=1,
    linestyle="--",
    linewidth=1
)


plt.xlabel(
    r"Frequency $\nu$ [Hz]",
    fontsize=15
)

plt.ylabel(
    r"$F_\nu(M) / F_\nu(1.4M_\odot)$",
    fontsize=15
)

plt.title(
    r"Flux Relative to $1.4\,M_\odot$ Model",
    fontsize=18
)

plt.legend(fontsize=11)

plt.minorticks_on()

plt.grid(
    True,
    which="both",
    ls="--",
    lw=0.5,
    alpha=0.4
)

plt.tick_params(
    axis="both",
    which="both",
    top=True,
    right=True,
    direction="in",
    length=5,
    width=1
)

ax = plt.gca()

ax.xaxis.set_major_locator(
    ticker.LogLocator(base=10, numticks=6)
)

ax.xaxis.set_major_formatter(
    ticker.LogFormatterSciNotation(base=10)
)

ax.tick_params(
    axis="both",
    labelsize=11
)

plt.tight_layout()

plt.show()

# 1.4 case is at one and you can see everything else a lot better!

In [ ]:
# making a general case

def compare_ns_masses(
        masses,
        frequencies,
        Mdot,
        R_in,
        R_out,
        inclination,
        D
):

    plt.figure(figsize=(8, 6))

    for mass in masses:
        M = mass * M_sun
        fluxes = []
        for nu in frequencies:
            F = disk_flux_nu(
                nu * u.Hz,
                M,
                Mdot,
                R_in,
                R_out,
                inclination,
                D
            )
            fluxes.append(F)

        # convert from W/m^2/Hz to erg/s/cm^2/Hz
        fluxes_cgs = np.array([
            f.to_value(
                u.erg / (u.cm**2 * u.s * u.Hz)
                ) 
                for f in fluxes
        ])

        plt.loglog(
            frequencies,
            fluxes_cgs,
            label=fr"$M = {mass}\,M_\odot$",
            linewidth=2
        )

    plt.xlabel(r"Frequency $\nu$ [Hz]", fontsize=15)
    plt.ylabel(
        r"$F_\nu$ [erg cm$^{-2}$ s$^{-1}$ Hz$^{-1}$]",
        fontsize=15
    )

    plt.title(
        "Accretion Disk Spectra for Different NS Masses",
        fontsize=18
    )

    plt.grid(
        True,
        which="both",
        ls="--",
        lw=0.5,
        alpha=0.4
    )

    plt.legend()

    plt.tight_layout()

    plt.show()

#ex

masses = [1.2, 1.4, 1.8, 2.0]

frequencies = np.logspace(14, 19, 200)

compare_ns_masses(
    masses,
    frequencies,
    Mdot,
    R_in,
    R_out,
    inclination,
    D
)


